# Experiment 1: Time and Frequency Domain, and the Pendulum
**Course:** CSAI4014 Algorithms for Intelligent Systems and Robotics  
**Student:** Aadi Chopra | SAP ID: 500120163 | Roll No: R2142231902 | Batch: B2 AIML  
**Repository:** https://github.com/aadichopra/aisr-lab-500120163  

## 1. Aim
To represent a sampled multi-tone continuous signal in the time domain and frequency domain using the Discrete Fourier Transform, demonstrate frequency aliasing under sub-Nyquist sampling, and simulate the nonlinear dynamics of a simple pendulum under various viscous damping regimes.


In [ ]:
import os
import numpy as np
import scipy.integrate as integrate
import matplotlib.pyplot as plt

os.makedirs('figs', exist_ok=True)

# 1. Signal Parameters
fs = 400.0   # Sampling rate (Hz)
T = 2.5      # Record length (s)
dt = 1.0 / fs
t = np.arange(0, T, dt)
N = len(t)
df = fs / N
print(f"Sampling Parameters: fs = {fs} Hz, T = {T} s, N = {N}, dt = {dt*1000:.2f} ms, df = {df:.2f} Hz, Nyquist = {fs/2:.1f} Hz")

# Synthesize multi-tone signal with Gaussian noise
rng = np.random.default_rng(101)
noise = 0.25 * rng.standard_normal(N)
x = (1.20 * np.sin(2 * np.pi * 6.0 * t) + 
     0.65 * np.sin(2 * np.pi * 18.0 * t) + 
     0.30 * np.sin(2 * np.pi * 45.0 * t) + 
     noise)

# Plot Figure 1.1: Time Domain Signal
fig, ax = plt.subplots(figsize=(6.5, 3.2), dpi=150)
ax.plot(t[:200], x[:200], color='#1f77b4', linewidth=1.2, label='Sampled Signal x(t)')
ax.set_xlabel('Time t (seconds)')
ax.set_ylabel('Amplitude (a.u.)')
ax.set_title('Synthesized 3-Tone Noisy Signal (First 0.50 s, fs = 400 Hz)', fontsize=11)
ax.grid(True, linestyle='--', alpha=0.5)
ax.legend(loc='upper right')
plt.tight_layout()
plt.savefig('figs/fig1_1_signal.png')
plt.show()


In [ ]:
# 2. Discrete Fourier Transform (One-Sided Spectrum)
X = np.fft.rfft(x)
f = np.fft.rfftfreq(N, d=dt)
amp = np.abs(X) * 2.0 / N
amp[0] /= 2.0

for target_f, true_a in [(6.0, 1.20), (18.0, 0.65), (45.0, 0.30)]:
    idx = int(round(target_f / df))
    print(f"Set: {target_f:4.1f} Hz -> Measured: {f[idx]:4.1f} Hz, Amp: {amp[idx]:.3f} (True: {true_a:.2f})")

# Plot Figure 1.2: One-Sided Spectrum
fig, ax = plt.subplots(figsize=(6.5, 3.2), dpi=150)
ax.plot(f, amp, color='#2ca02c', linewidth=1.2)
ax.set_xlim(0, 80)
ax.set_xlabel('Frequency (Hz)')
ax.set_ylabel('Single-Sided Amplitude')
ax.set_title('Discrete Fourier Transform Spectrum (Resolved Peaks at 6, 18, and 45 Hz)', fontsize=11)
ax.grid(True, linestyle='--', alpha=0.5)
for target_f in [6.0, 18.0, 45.0]:
    idx = int(round(target_f / df))
    ax.annotate(f"{f[idx]:.1f} Hz ({amp[idx]:.2f})", xy=(f[idx], amp[idx]),
                xytext=(f[idx] + 1.5, amp[idx] + 0.10),
                arrowprops=dict(facecolor='black', shrink=0.05, width=1, headwidth=4))
plt.tight_layout()
plt.savefig('figs/fig1_2_spectrum.png')
plt.show()


In [ ]:
# 3. Aliasing Demonstration: 14 Hz sampled at 20 Hz
fs_low = 20.0
t_low = np.arange(0, T, 1.0 / fs_low)
N_low = len(t_low)
x_low = np.sin(2 * np.pi * 14.0 * t_low)
X_low = np.fft.rfft(x_low)
f_low = np.fft.rfftfreq(N_low, d=1.0 / fs_low)
amp_low = np.abs(X_low) * 2.0 / N_low

apparent_alias_f = f_low[np.argmax(amp_low[1:]) + 1]
print(f"Aliasing: 14 Hz sampled at {fs_low} Hz -> Apparent alias at {apparent_alias_f:.1f} Hz")

# Plot Figure 1.3: Aliasing Demonstration
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(6.5, 4.2), dpi=150)
t_fine = np.linspace(0, 0.8, 500)
ax1.plot(t_fine, np.sin(2 * np.pi * 14.0 * t_fine), 'gray', linestyle=':', label='True 14 Hz signal')
ax1.plot(t_low[:17], x_low[:17], 'ro-', label='Sampled at 20 Hz')
ax1.plot(t_fine, -np.sin(2 * np.pi * 6.0 * t_fine), 'b--', label='Apparent 6 Hz alias waveform')
ax1.set_xlabel('Time (s)'); ax1.set_ylabel('Amplitude')
ax1.legend(loc='upper right', fontsize=8)
ax1.set_title('Time Domain Aliasing: 14 Hz Sinusoid Undersampled at 20 Hz Appears as 6 Hz', fontsize=10)
ax1.grid(True, linestyle='--', alpha=0.5)

ax2.stem(f_low, amp_low, linefmt='r-', markerfmt='ro', basefmt='k-')
ax2.set_xlabel('Frequency (Hz)'); ax2.set_ylabel('Amplitude')
ax2.set_title('DFT Spectrum of Undersampled Signal Showing False Peak at 6.0 Hz', fontsize=10)
ax2.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig('figs/fig1_3_aliasing.png')
plt.show()


In [ ]:
# 4. Simple Pendulum Numerical Integration (L = 0.85 m)
g, L = 9.81, 0.85
T0 = 2 * np.pi * np.sqrt(L / g)
print(f"Small-angle linear period T0: {T0:.4f} s")

def pendulum_ode(t, state, b):
    theta, omega = state
    return [omega, -b * omega - (g / L) * np.sin(theta)]

t_eval = np.linspace(0, 10, 1000)
theta_init = np.radians(25.0)
sol_undamped = integrate.solve_ivp(pendulum_ode, (0, 10), [theta_init, 0.0], args=(0.0,), t_eval=t_eval)
sol_under = integrate.solve_ivp(pendulum_ode, (0, 10), [theta_init, 0.0], args=(0.35,), t_eval=t_eval)
sol_over = integrate.solve_ivp(pendulum_ode, (0, 10), [theta_init, 0.0], args=(3.50,), t_eval=t_eval)

# Plot Figure 1.4: Damping Regimes
fig, ax = plt.subplots(figsize=(6.5, 3.4), dpi=150)
ax.plot(t_eval, np.degrees(sol_undamped.y[0]), label='Undamped (b = 0.0 N·s/m)', color='#1f77b4', linewidth=1.5)
ax.plot(t_eval, np.degrees(sol_under.y[0]), label='Underdamped (b = 0.35 N·s/m)', color='#2ca02c', linewidth=1.5)
ax.plot(t_eval, np.degrees(sol_over.y[0]), label='Overdamped (b = 3.50 N·s/m)', color='#d62728', linewidth=1.5)
ax.set_xlabel('Time t (seconds)')
ax.set_ylabel('Angular Displacement θ (degrees)')
ax.set_title(f'Simple Pendulum Trajectories under Damping Regimes (L = {L} m, $\\theta_0 = 25^\\circ$)', fontsize=11)
ax.legend(loc='upper right')
ax.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig('figs/fig1_4_pendulum_damping.png')
plt.show()


In [ ]:
# 5. Phase Portrait and Nonlinear Period Lengthening
amplitudes_deg = [15, 30, 45, 60, 75, 90]
measured_periods = []

for amp_deg in amplitudes_deg:
    theta0 = np.radians(amp_deg)
    sol = integrate.solve_ivp(pendulum_ode, (0, 20), [theta0, 0.0], args=(0.0,), t_eval=np.linspace(0, 20, 10000), rtol=1e-8, atol=1e-8)
    signs = np.sign(sol.y[0])
    crossings = np.where(np.diff(signs) > 0)[0]
    periods = np.diff(sol.t[crossings])
    measured_periods.append(float(np.mean(periods)))
    print(f"Angle: {amp_deg:2d} deg -> Period: {measured_periods[-1]:.4f} s (Ratio T/T0: {measured_periods[-1]/T0:.4f})")

# Plot Figure 1.5: Phase Portrait and Nonlinear Period
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(7.2, 3.3), dpi=150)
for amp_deg in [15, 45, 90]:
    sol = integrate.solve_ivp(pendulum_ode, (0, 10), [np.radians(amp_deg), 0.0], args=(0.0,), t_eval=np.linspace(0, 10, 1000))
    ax1.plot(sol.y[0], sol.y[1], label=f'$\theta_0 = {amp_deg}^\circ$')
ax1.set_xlabel('Displacement θ (rad)'); ax1.set_ylabel('Angular Velocity ω (rad/s)')
ax1.set_title('Phase Portrait (Undamped)', fontsize=10)
ax1.legend(loc='upper right', fontsize=8)
ax1.grid(True, linestyle='--', alpha=0.5)

ax2.plot(amplitudes_deg, measured_periods, 'bo-', label='Measured Period (RK45)')
ax2.axhline(T0, color='red', linestyle='--', label=f'Linear $T_0 = {T0:.4f}$ s')
ax2.set_xlabel('Initial Amplitude $\theta_0$ (degrees)')
ax2.set_ylabel('Period T (seconds)')
ax2.set_title('Nonlinear Period Dilation', fontsize=10)
ax2.legend(loc='upper left', fontsize=8)
ax2.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig('figs/fig1_5_phase_portrait_period.png')
plt.show()
